# Recap - News Texts

In [1]:
import requests
import json
from bs4 import BeautifulSoup

page = BeautifulSoup(requests.get("https://www.tagesschau.de/").text, "html.parser") #visit the page

#<a class="teaser__link" -> this is the identified pattern that leads to all articles
articleList = page.find_all("a", class_="teaser__link", href=True)

baseUrl = "https://www.tagesschau.de" #the base url; needs to be combined with the found hrefs
articles = [] #empty list to store articles
for article in articleList:
  if article["href"].startswith("/") and not article["href"].startswith("/multimedia"): #filter out unwanted elements that might cause problems
    articlePage = BeautifulSoup(requests.get(baseUrl + article["href"]).text,"html.parser") #gathering the data
    article = articlePage.find("script", type="application/ld+json").text
    article = json.loads(article)['articleBody']
    articles.append(article)

articles[0]

'US-Präsident Trump hat Treibhausgase für unproblematisch erklärt - und damit die zentrale rechtliche Grundlage für Klimagesetze in den USA gekippt. Ein Überblick über weltweite Folgen und Kritik.   Was hat Trump entschieden? US-Präsident Donald Trump hat eine der wichtigsten Vorgaben für den Klimaschutz in den USA abgeschafft. Er strich das sogenannte Endangerment Finding, mit dem die US-Umweltbehörde die wichtigsten Treibhausgase - darunter CO2 - seit 2009 als Gefahr für die öffentliche Gesundheit und das Wohlergehen eingestuft hat. Die bisherige Einstufung der Treibhausgase als gefährlich war die Grundlage dafür, dass die Umweltbehörde Grenzen für solche Schadstoffe festlegen durfte.    Wie schlimm sind Treibhausgase? In der Wissenschaft gibt es seit langem keinen Zweifel daran, dass Treibhausgase die Haupttreiber der sich zuspitzenden Klimakrise sind. Neue Erkenntnisse, die diese Einschätzung ins Wanken bringen könnten, gibt es nicht. Laut dem Weltklimarat (IPCC) haben Treibhausgas



---


# Spacy

https://spacy.io/

The spaCy library for Python is a powerful Natural Language Processing (NLP) library that provides fast, efficient, and production-ready tools for processing and analyzing text data. Here's an overview of its abilities:

**1. Text Preprocessing**

- Tokenization: Splits text into individual words, punctuation, or other meaningful units (tokens).

- Lemmatization: Reduces words to their base forms (e.g., "running" → "run").

**2. Part-of-Speech (POS) Tagging**

- Identifies the grammatical roles of words in a sentence (e.g., noun, verb, adjective).

**3. Named Entity Recognition (NER)**

- Detects named entities like people, organizations, locations, dates, monetary values, etc., in text.

**4. Dependency Parsing**

- Analyzes the syntactic structure of sentences and identifies relationships between words.

**5. Sentence Boundary Detection**

- Identifies sentence boundaries in a block of text.

**6. Word and Sentence Embeddings**

- Provides word and sentence vector representations using pre-trained embeddings or custom models.

**7. Similarity and Semantic Analysis**

- Computes similarity scores between words, phrases, or documents using embeddings.

**8. Support for Multiple Languages**

- Provides models for many languages, with pre-trained pipelines for tasks like NER, POS tagging, and parsing.

**9. Training and Fine-tuning**

- Supports training custom NLP models, including NER, POS tagging, and text classification.

**10. Visualization**

- Provides tools like displaCy to visualize parse trees, entities, and more.

The abilities listed above are implemented as Pipelines, which are included in models:

https://spacy.io/models/de#de_core_news_lg

For instance, the german model *de_core_news_lg* implements:

- tok2vec
- tagger
- morphologizer
- parser
- lemmatizer
- attribute_ruler
- ner

In order to use a model, it first needs to be downloaded:

In [3]:
!python -m spacy download de_core_news_lg

/Users/hd/Desktop/Machine Learning/.venv/bin/python: No module named spacy




---


# Basic utilities

After downloading, the model's abilities can be used by initializing a pipeline using the *load()* method.

In [4]:
import spacy

nlp = spacy.load("de_core_news_lg")
type(nlp)

ModuleNotFoundError: No module named 'spacy'

Now, texts can be passed through the pipeline by just passing them to the variable:

In [ ]:
doc = nlp(articles[0])
type(doc)

The resulting variable is of the type *Doc* and allows for accessing the tokens within the document and their corresponding annotations (https://spacy.io/api/doc)

Splitting the document on sentence level:

In [ ]:
#reading the document sentence by sentence:
for sentence in doc.sents:
  print (sentence)

Iterating over individual tokens:

In [ ]:
for token in doc[:20]: #print 20 tokens
  print (token)



---

# Lemmatization

"Lemmatization (or less commonly lemmatisation) in linguistics is the process of grouping together the inflected forms of a word so they can be analysed as a single item, identified by the word's lemma, or dictionary form." (https://en.wikipedia.org/wiki/Lemmatization)

Examples:

Token|Lemma
---|---
best|good
driving|(to) drive

In [ ]:
for token in doc[:20]: #print 20 tokens
  print (token.lemma_)



---
# Part-of-Speech

The term *Part of speech* describes a word class, like nouns, verbs, adjectives, etc.

The process of assigning part-of-speeches to tokens is called tagging (or POS tagging). Part of speeches can be accessed via the *.pos_* property of a token:

In [ ]:
for token in doc[:20]: #print 20 tokens
  print (f"{token}\t{token.pos_}")

For an explanation of the individual tags, it is possible to use the *explain()* method for a given tag:

In [ ]:
for token in doc[:20]: #print 20 tokens
  print (f"{token.pos_}\t{spacy.explain(token.pos_)}")



---

# Named Entity Recognition

Describes the process of identifying and categorizing entities in a text. These can be Persons, Locations, Organisations, and so on.
Similar to POS-tagging, the process of assigning an entity type to a token is called tagging.

In [ ]:
for token in doc[:20]: #print 20 tokens
  print (f"{token.lemma_}\t{token.ent_type_}")

In [ ]:
for entity in doc.ents: #here, we operate on document level
  print (f"{entity}\t{entity.label_}")

In [ ]:
for entity in doc.ents: #here, we operate on document level
  print (f"{entity.label_}\t{spacy.explain(entity.label_)}")



---

#Morphological Features
Aside from POS and NER tags, it is also possible to display morphological features like *genus*, *numerus*, *kasus*.

In [ ]:
for token in doc[:20]: #print 20 tokens
  print (f"{token}\t{token.morph}")



---
#Dependency Trees

Dependency trees allow to analyze the syntactical structure of sentences. They can be visualized using the *displacy* submodule from spacy.


In [ ]:
from spacy import displacy

for sentence in doc.sents:
  displacy.render(sentence, style="dep")
  break #abort after the first



---
#Word similarity

The *tok2vec* part of the pipeline allows for calculating distance measures between words and sentences. Without going into the details of it, these similarity measures are based on numerical word representations.

Comparing the similarity of sentences:


In [2]:
sent1 = nlp("Ich liebe Burger")
sent2 = nlp("Ich liebe Äpfel")
sent3 = nlp("Er macht Pause")

print (sent1.similarity(sent2))
print (sent1.similarity(sent3))


NameError: name 'nlp' is not defined



---
#Most frequent entity types in the news today

 - Send all articles through the pipeline
 - Retrieve all NER annotations
 - Count the number of appearances


In [ ]:
nerDict = {}

for article in articles:
  doc = nlp(article)
  for entity in doc.ents:
    if entity.label_ in nerDict:
      nerDict[entity.label_] += 1
    else:
      nerDict[entity.label_] = 1

nerDict

#Most frequently mentioned person in the news today

 - iterate over articles
 - annotate articles with ner tags
 - find all tokens that belong to the *PER* entity type

In [ ]:
personDict = {}

for article in articles:
  doc = nlp(article)
  for entity in doc.ents:
    if entity.label_ == "PER":
      if entity.lemma_.split()[-1] in personDict: #only last name if applicable
        personDict[entity.lemma_.split()[-1]] += 1
      else:
        personDict[entity.lemma_.split()[-1]] = 1

In [ ]:
#order personDict by value

for key, value in sorted(personDict.items(), key=lambda item: item[1], reverse=True):
  if value > 1:
    print(f"{key}: {value}")

Now let us compare this to the most frequently mentioned person in a russian news outlet:

In [ ]:
baseUrl = "https://www.vesti.ru"

page = BeautifulSoup(requests.get("https://www.vesti.ru/news").text, "html.parser") #visit the page
links = page.find_all("a",href=True)
articleLinks = []
for link in links:
  if link["href"].startswith("/article/"):
    if link["href"] not in articleLinks:
      articleLinks.append(link["href"])

articlesRU = []
for link in articleLinks:
  currentPage = BeautifulSoup(requests.get(baseUrl+link).text, "html.parser") #visit the page
  text = currentPage.find_all("div",class_="article__text")
  articleText = ""
  for element in text:
    articleText += element.text
  if not len(articleText) == 0:
    articlesRU.append(articleText.replace("\n",""))

In [ ]:
articlesRU[0]

In [ ]:
!python -m spacy download ru_core_news_lg

In [ ]:
nlpRu = spacy.load("ru_core_news_lg")

In [ ]:
personDictRU = {}

for article in articlesRU:
  doc = nlpRu(article)
  for entity in doc.ents:
    if entity.label_ == "PER":
      if entity.lemma_.split()[-1] in personDictRU: #only last name if applicable
        personDictRU[entity.lemma_.split()[-1]] += 1
      else:
        personDictRU[entity.lemma_.split()[-1]] = 1

In [ ]:
for key, value in sorted(personDictRU.items(), key=lambda item: item[1], reverse=True):
  if value > 1:
    print(f"{key}: {value}")